# Generative AI 005 - Working with Models

The lesson's own code needs a paid API key on every line. **This notebook needs
none.** Three of its claims can be checked exactly on a laptop:

| Part | What we check |
|---|---|
| A | an LLM returns `str`; a chat model returns a 9-field `AIMessage` |
| B | temperature is one division - and what that division does |
| C | the cost of creativity, in wrong answers |
| D | what narrowing an embedding vector actually loses |
| E | what a million chunks costs to store |

Part A needs `langchain-core`. Parts B-E need `numpy`, `scipy` and
`scikit-learn`.

## Part A - String in / string out, against messages in / message out

The split that confuses everyone reading LangChain code for the first time: some
examples print `result` and some print `result.content`. Here is why, checked
with LangChain's own stub models - which have the real interfaces and need no
key at all.

In [ ]:
from langchain_core.language_models.fake import FakeListLLM
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import Runnable

answer = "The capital of India is New Delhi."

In [ ]:
# An LLM: string in, string out.
llm = FakeListLLM(responses=[answer])
out = llm.invoke("What is the capital of India?")

print("type   :", type(out).__name__)
print("value  :", repr(out))
assert isinstance(out, str)

# Nothing to unwrap. There is also nowhere to put a role, a tool call
# or a token count.

In [ ]:
# A chat model: messages in, a message OBJECT out.
chat = FakeListChatModel(responses=[answer])
out = chat.invoke("What is the capital of India?")

print("type    :", type(out).__name__)
print("content :", out.content)
fields = sorted(type(out).model_fields)
print("fields  :", len(fields))
for f in fields:
    print("   ", f)

assert type(out).__name__ == "AIMessage"

Nine fields. That is the answer to the question: an LLM hands you the answer, a
chat model hands you a **box** with the answer in it - plus room for which tools
it wants called, how many tokens the call cost, and the provider's raw response.

None of that fits in a string, which is a large part of why chat models won.

In [ ]:
# Roles - the thing an LLM has no place for.
out = chat.invoke([
    SystemMessage("You are a geography teacher. Answer in one short sentence."),
    HumanMessage("What is the capital of India?"),
])
print(type(out).__name__, "|", out.content)

# CAREFUL: a stub model ignores the system prompt. This shows that the
# INTERFACE accepts roles. It does NOT show that roles change the answer -
# that needs a real model and a real key.

In [ ]:
# And why .invoke is spelled the same way everywhere in LangChain:
print(isinstance(llm, Runnable), isinstance(chat, Runnable))
# Prompts and chains implement Runnable too. That is why chain.invoke(...)
# will look identical when you meet it.

## Part B - Temperature, properly

"Temperature controls creativity" does not tell you what number to pick.
Temperature has an exact meaning: the model has a **score** for every word it
might say next, and it turns those into probabilities with a softmax. Temperature
is what the scores are **divided by** first.

$$P(w_i) = \frac{\exp(s_i / T)}{\sum_j \exp(s_j / T)}$$

Small `T` stretches the gaps between scores, so the favourite dominates. Large
`T` squeezes them together, so the tail gets a chance.

In [ ]:
import numpy as np

# Eight candidate next words after "The capital of India is".
# INVENTED - reading a real model's scores needs a model - but shaped like
# a real distribution: one clear favourite and a weak tail.
words  = ["New", "Delhi", "located", "a", "the", "Mumbai", "beautiful", "Kolkata"]
scores = np.array([6.0, 5.4, 3.1, 2.8, 2.6, 1.9, 1.2, 0.4])


def softmax(z, t):
    z = np.asarray(z, float) / t     # <- the whole of temperature is this line
    z = z - z.max()                  # numerical safety only, changes nothing
    e = np.exp(z)
    return e / e.sum()

In [ ]:
rng = np.random.default_rng(0)

print(f"{'T':>5} {'p(top)':>8} {'p(2nd)':>8} {'entropy':>9} "
      f"{'effective':>10} {'distinct/2000':>14}")
for t in (0.1, 0.3, 0.7, 1.0, 1.5, 2.0):
    p = softmax(scores, t)
    order = np.argsort(p)[::-1]
    entropy = -(p * np.log(p + 1e-12)).sum()
    effective = np.exp(entropy)      # "how many words is it really choosing between"
    distinct = len(np.unique(rng.choice(len(p), size=2000, p=p)))
    print(f"{t:>5.1f} {p[order[0]]:>8.4f} {p[order[1]]:>8.4f} {entropy:>9.4f} "
          f"{effective:>10.2f} {distinct:>14}")

Read the `effective` column - it is `exp(entropy)`, and it reads as "how many
words is the model really choosing between".

At `T=0.1` it is **1.02**: deterministic in everything but name, which is exactly
what you want for code and arithmetic. At `T=2.0` it is **5.32** of the eight.

The last column is the check: the sampling agrees with the arithmetic. At low
temperature only 2 different words ever appear in 2000 draws; from `T=1.0` all 8 do.

In [ ]:
p_low, p_high = softmax(scores, 0.1), softmax(scores, 2.0)
print(f"T=0.1  top word takes {p_low.max():.4f}")
print(f"T=2.0  top word takes {p_high.max():.4f}")

assert abs(p_low.max() - 0.9975) < 1e-4
assert abs(p_high.max() - 0.3788) < 1e-4

## Part C - What creativity costs

Two of the eight candidates - "Mumbai" and "Kolkata" - are simply **wrong
answers**. Turning the temperature up gives them probability.

In [ ]:
wrong = [words.index("Mumbai"), words.index("Kolkata")]

print(f"{'T':>5} {'p(wrong answer)':>18}")
for t in (0.1, 0.3, 0.7, 1.0, 1.5, 2.0):
    p = softmax(scores, t)
    print(f"{t:>5.1f} {p[wrong].sum():>18.6f}")

r = softmax(scores, 2.0)[wrong].sum() / softmax(scores, 0.7)[wrong].sum()
print(f"\nFrom T=0.7 to T=2.0 the risk grows {r:.0f}x")

About **33x**. This is why the standard advice splits by task instead of giving
one number:

| Temperature | Use it for |
|---|---|
| 0 - 0.3 | code, maths, extraction, anything factual |
| 0.5 - 0.7 | ordinary questions and answers |
| 0.9+ | stories, brainstorming - where being wrong is not a failure |

> **What this figure is.** The eight scores are invented. The softmax and the
> sampling are **exact**. So this measures what temperature does to a
> distribution - a property of the formula that holds for any scores - and not
> the behaviour of any particular model.

## Part D - How many dimensions does a vector need?

Embedding models turn text into a vector, and `dimensions` sets how long it is.
More dimensions hold more detail and cost more, on disk *and* on every query. So
how few can you get away with?

In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CORPUS = [
    "Virat Kohli is an Indian batter known for chasing targets in one day cricket.",
    "Jasprit Bumrah is a fast bowler famous for yorkers at the death overs.",
    "Rohit Sharma is an opening batter who has scored three double centuries.",
    "MS Dhoni was a wicketkeeper and captain known for finishing close matches.",
    "New Delhi is the capital of India and the seat of the national government.",
    "Paris is the capital of France and sits on the river Seine.",
    "Tokyo is the capital of Japan and one of the largest cities in the world.",
    "Kolkata is the capital of the Indian state of West Bengal.",
    "Linear regression fits a straight line by minimising the squared error.",
    "A decision tree splits the data repeatedly to reduce impurity at each node.",
    "Gradient descent updates the weights in the direction that lowers the loss.",
    "A neural network stacks layers of weighted sums and non linear activations.",
]

QUERIES = [
    "Who is good at chasing a target in cricket?",
    "What is the capital city of France?",
    "How does gradient descent work?",
    "Which bowler bowls yorkers?",
    "Tell me about decision trees.",
    "What is the capital of India?",
]

In [ ]:
vec = TfidfVectorizer(stop_words="english")
docs_full = vec.fit_transform(CORPUS).toarray()
q_full = vec.transform(QUERIES).toarray()
full_dim = docs_full.shape[1]

sims_full = cosine_similarity(q_full, docs_full)
top_full = sims_full.argmax(axis=1)

print(f"full width: {full_dim} dimensions")
for q, i in zip(QUERIES, top_full):
    print(f"  {q:<45} -> doc {i}")

In [ ]:
top3_full = [set(np.argsort(sims_full[i])[::-1][:3]) for i in range(len(QUERIES))]

print(f"{'dims':>5} {'bytes/doc':>10} {'top-1':>8} {'top-3 overlap':>14}")
for d in (2, 4, 8, 16, 32, 64, full_dim):
    if d >= full_dim:
        docs_r, q_r, d = docs_full, q_full, full_dim
    else:
        svd = TruncatedSVD(n_components=d, random_state=0)
        docs_r = svd.fit_transform(docs_full)
        q_r = svd.transform(q_full)
    sims = cosine_similarity(q_r, docs_r)
    agree = int((sims.argmax(axis=1) == top_full).sum())
    overlap = np.mean([len(top3_full[i] & set(np.argsort(sims[i])[::-1][:3])) / 3
                       for i in range(len(QUERIES))])
    print(f"{d:>5} {d*4:>10} {agree:>4}/{len(QUERIES)} {overlap:>14.2f}")

The **top-1** column behaves the way you would hope. Too narrow and answers get
lost; from **16 dimensions** - one fifth of the full width - every query comes
back correct.

The **top-3** column does not behave. It wobbles between 0.56 and 0.67 and never
climbs. That is worth understanding rather than ignoring, so check the reason
directly.

In [ ]:
zeros = (sims_full == 0).sum()
total = sims_full.size
print(f"{zeros} of {total} full-width similarities are EXACTLY zero "
      f"({100*zeros/total:.0f}%)")
print()
print("A short query shares no word at all with most of the corpus, so the")
print("reference 'top three' is largely an arbitrary pick out of a big tie.")
print("The narrowed vectors are dense and break that tie differently, so the")
print("top-3 column is measuring TIE-BREAKING, not lost accuracy.")
print()
print("Trust the top-1 column. Noticing a metric that measures nothing is a")
print("more useful habit than reading a clean table.")

> **And what this experiment is not.** It uses TF-IDF vectors narrowed with SVD,
> because no embedding model runs offline here. TF-IDF compares **words**, not
> meaning. The *shape* of the trade-off is real - narrow too far and retrieval
> breaks, and past some width nothing improves - but "16" is a fact about this
> corpus and this method, **not** a setting to use with a real embedding model.

## Part E - What it costs to store

This part needs no caveats at all. It is multiplication.

In [ ]:
def gigabytes(dims, n_chunks, bytes_per_number=4):
    return dims * bytes_per_number * n_chunks / 1024 ** 3


for d, label in ((3072, "text-embedding-3-large, default"),
                 (1536, "text-embedding-3-small, default"),
                 (384, "all-MiniLM-L6-v2, local"),
                 (300, "the lesson's mini-project"),
                 (32, "the lesson's first example")):
    print(f"{d:>5} dims  {gigabytes(d, 1_000_000):>7.2f} GB   ({label})")

assert abs(gigabytes(3072, 1_000_000) - 11.44) < 0.01

In [ ]:
# And it is paid TWICE: on disk, and on every single query, because the
# question vector is compared against every stored vector.
n = 1_000_000
for d in (3072, 384):
    print(f"{d:>5} dims -> {d * n / 1e9:.1f} billion multiply-adds per query "
          f"in a brute-force search")
print()
print("Real vector databases index rather than scanning everything, but the")
print("width still shows up in the index size and in the comparison cost.")

## What to take away

- An LLM returns a `str`. A chat model returns an **`AIMessage` with 9 fields**,
  and that is why chat code reads `.content`.
- Both are `Runnable`s, which is why `.invoke` is spelled the same everywhere.
- **Temperature is one division inside a softmax.** It took the top word from
  **0.9975** of the probability at `T=0.1` to **0.3788** at `T=2.0`.
- **Creativity costs accuracy**: the chance of a wrong answer rose about **33x**
  between `T=0.7` and `T=2.0`.
- **`dimensions` is a real trade-off** - 16 of 82 was enough here - and a
  million chunks is **11.44 GB** at 3072 dimensions against **0.12 GB** at 32.

## Exercises

1. Change the scores so the top two are nearly tied (say 6.0 and 5.9). Redo the
   temperature table. At which temperature does the favourite stop winning most
   of the time, and why is that different from the original?
2. `effective = exp(entropy)` was used as "how many words is it choosing
   between". Verify that intuition: for a uniform distribution over `k` words,
   show that it returns exactly `k`.
3. Greedy decoding is `T -> 0`. Take the limit numerically (try `T = 0.01`,
   `0.001`) and watch. What goes wrong in the arithmetic, and what does the
   `z - z.max()` line in `softmax` have to do with it?
4. Add four more documents to `CORPUS` on a *new* topic and two matching
   queries. Does 16 dimensions still recover every answer? Would you expect the
   needed width to grow with the number of documents, the number of topics, or
   neither?
5. Replace `float32` with `float16` in the storage arithmetic. How much do you
   save, and what would you want to measure before shipping that change?
6. If you have an API key: run the real thing. Embed the corpus with
   `OpenAIEmbeddings(model="text-embedding-3-large", dimensions=d)` for `d` in
   32, 256, 3072, and see at which width the retrieval stops improving. That is
   the measurement this notebook could not make.